In [631]:
import json
from pathlib import Path
import os
import re
from json import JSONDecodeError
from typing import Optional
    
class GeneralTree:
    def __init__(self, data):
        self.data = data
        self.children = []
        self.parent = None
        
    def add_child(self, data):
        self.children.append(data)
        data.parent = self
        
    def get_level(self):
        level = 0
        itr = self.parent
        while itr:
            itr = itr.parent
            level += 1
        return level
    
    def print_tree(self):
        spaces = " " * self.get_level() * 2
        prefix = spaces + "|--" if self.parent else ""
        print(prefix+self.data)
        if self.children:
            for child in self.children:
                child.print_tree()
                
    def get_root(self, rotation: Optional[bool]=False):
        if self:
            if self.parent is None:
                return self.data
            else:
                if not rotation:
                    print("Not a root node!")
                return
        else:
            if not rotation:
                print("General Tree is empty!")
            return
        
    def is_root(self):
        if self:
            if self.parent is None:
                return True
  
        return False
        
    def count_items(self):
        if not self:
            print("General Tree is empty!")
            return 0
        
        count = 1
        if self.children:
            for child in self.children:
                count += child.count_items()
        return count
    
    def get_leaf_nodes(self, leaves=None):
        if not self:
            print("General Tree is empty!")
            return []
                
        if leaves is None:
            leaves = []
        
        if not self.children:
            leaves.append(self.data)
        
        if self.children:
            for child in self.children:
                child.get_leaf_nodes(leaves)
                
        return leaves
    
    # using nodes as the reference and not edges
    def get_depth(self):
        if not self:
            print("General Tree is empty!")
            return 0
        
        depth = 0
        if self.children:
            for child in self.children:
                depth = max(child.get_depth(), depth)
        return depth + 1
    
    # using nodes as the reference and not edges
    def get_height(self):
        if not self:
            print("General Tree is empty!")
            return 0
                
        height = 0
        if self.children:
            for child in self.children:
                height = max(child.get_height(), height)
        return height + 1
        
    def get_diameter(self):
        if not self:
            print("General Tree is empty!")
            return 0
        
        left, right = 0, 0
        if self.children:
            for child in self.children:
                node = child.get_depth()
                if node > left:
                    right, left, = left, node
                elif node > right:
                    right = node
                
        diameter = 0
        if self.children:
            for child in self.children:
                diameter = max(diameter, child.get_diameter())
                
        return max(diameter, left+right+1)
    
    def get_diameter_of(self, data, flag=False):
        diameter = None
        if not self:
            print("General Tree is empty!")
            return diameter
        
        if self.data == data:
            return self.get_diameter() 
        
        if self.children:
            for child in self.children:
                diameter = child.get_diameter_of(data, flag=True)
                if diameter is not None:
                    return diameter
        
        if not flag:
            print(f"No match found for {data}")
        
        return diameter    
    
    def depth_of(self, data, flag=False):
        depth = None
        if not self:
            print("General Tree is empty!")
            return depth
        
        if self.data == data:
            return self.get_level()
        
        if self.children:
            for child in self.children:
                depth = child.depth_of(data, flag=True)
                if depth is not None:
                    return depth

        if not flag:
            print(f"No match found for {data}")
            
        return depth
        
    def height_of(self, data, flag=False):
        height = None
        if not self:
            print("General Tree is empty!")
            return height
        
        if self.data == data:
            return self.get_height() - 1
        
        if self.children:
            for child in self.children:
                height = child.height_of(data, flag=True)
                if height is not None:
                    return height
        
        if not flag:
            print(f"No match found for {data}")    
        
        return height
    
    def get_leaf_nodes_count(self):
        return len(self.get_leaf_nodes())
    
    def get_parent(self, data, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return None
        
        if is_found is None:
            is_found = [False]
        
        parent = None
        if self.data == data:
            is_found[0] = True
            if not self.is_root():
                if self.parent:
                    return self.parent.data
            
            print(f"{data} is the root node and has no parent!")
            return
            
        if self.children:
            for child in self.children:
                parent = child.get_parent(data, is_found, flag=True)
                if parent or is_found[0]:
                    if parent is None:
                        return
                    return parent
                
        if not flag:
            print(f"No match found for {data}") 
            
        return parent
            
    def get_parents(self, data, parents=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return []
        
        if parents is None:
            parents = []
            
        # mutable list/dictionary can have shared values across every level of recursion 
        if is_found is None:
            is_found = [False]
        
        if self.data == data:
            is_found[0] = True
            if not self.is_root():
                if self.parent:
                    parents.append(self.parent.data)
            else:  
                print(f"{data} is the root node and has no parent!")

            
        if self.children:
            for child in self.children:
                child.get_parents(data, parents, is_found, flag=True)
                
        if not flag and not is_found[0]:
            print(f"No match found for {data}")
            
        if not parents:
            return
            
        return parents
    
    def get_children(self, data, is_found=None, flag=False):
        children = None
        if not self:
            print("General Tree is empty!")
            return children
        
        if is_found is None:
            is_found = [False]
        
        if self.data == data:
            is_found[0] = True
            if self.children:
                children = []
                for child in self.children:
                    children.append(child.data)
                return children
            else:
                print(f"Match found, but no child available for {data}!")               
                return
                    
        if self.children:
            for child in self.children:
                children = child.get_children(data, is_found, flag=True)
                if children or is_found[0]:
                    if children is None:
                        return
                    return children
                
        if not flag and not is_found[0]:
            print(f"No match found for {data}") 
        
        return children
    
    def get_all_children(self, data, children=None, is_found=None, flag=False):
        if children is None:
            children = []
            
        # global keyword keeps the variable shared across every level of recursion but it stays the same throughtout even if we call the N number of times for a different use cases
        if is_found is None:
            is_found = [False]
            
        if not self:
            print("General Tree is empty!")
            return children
        
        if self.data == data:
            is_found[0] = True
            if self.children:
                for child in self.children:
                    children.append(child.data)
            else:
                print(f"Match found, but no child available for {data}!")
                    
        if self.children:
            for child in self.children:
                child.get_all_children(data, children, is_found, flag=True)
                
        if not flag and not is_found[0]:
            print(f"No match found for {data}") 
            
        if not children:
            return
        
        return children
    
    def get_grand_children(self, data, is_found=None, flag=False):
        grand_children = None
        if not self:
            print("General Tree is empty!")
            return grand_children
        
        if is_found is None:
            is_found = [False]
        
        if self.data == data:
            is_found[0] = True
            if self.children:
                for child in self.children:
                    if child.children:
                        grand_children = []
                        for grand_child in child.children:
                            grand_children.append(grand_child.data)
                        return grand_children
                    else:
                        print(f"Match found, but no grand children available for {data}!")
                        return
            else:
                print(f"Match found, but no children and grand children available for {data}!")
                return
        
        if self.children:
            for child in self.children:
                grand_children = child.get_grand_children(data, is_found, flag=True)
                if grand_children or is_found[0]:
                    if grand_children is None:
                        return
                    return grand_children
                
        if not flag and not is_found[0]:
            print(f"No match found for {data}") 
        
        return grand_children
    
    def get_all_grand_children(self, data, grand_children=None, is_found=None, flag=False): 
        if grand_children is None:
            grand_children = []
            
        if is_found is None:
            is_found = [False]
            
        if not self:
            print("General Tree is empty!")
            return
    
        if self.data == data:
            is_found[0] = True
            if self.children:
                for child in self.children:
                    if child.children:
                        for grand_child in child.children:
                            grand_children.append(grand_child.data)
                            
                if not grand_children:
                    print(f"Match found, but no grand children available for {data}!")
                        
            else:
                print(f"Match found, but no children and grand children available for {data}!")
                
        
        if self.children:
            for child in self.children:
                child.get_all_grand_children(data, grand_children, is_found, flag=True)
                
        if not flag and not is_found[0]:
            print(f"No match found for {data}") 
            
        if not grand_children:
            return
        
        return grand_children
    
    def get_grand_parent(self, data, is_found=None, flag=False):
        grand_parent = None
        if not self:
            print("General Tree is empty!")
            return grand_parent
        
        if is_found is None:
            is_found = [False]
        
        if self.data == data:
            is_found[0] = True
            if self.parent:
                if self.parent.parent:
                    return self.parent.parent.data
                print(f"{data} has no grand parent!")
                return 
            print(f"{data} is a root node and has no parent and grand parent!")
            return
            
        if self.children:
            for child in self.children:
                grand_parent = child.get_grand_parent(data, is_found, flag=True)
                if is_found[0] or grand_parent:
                    if grand_parent is None:
                        return
                    return grand_parent
                
        if not flag and not is_found[0]:
            print(f"No match found for {data}") 
            
        return grand_parent
    
    def get_all_grand_parent(self, data, grand_parents=None, is_found=None, flag=False):
        if grand_parents is None:
            grand_parents = []
            
        if is_found is None: 
            is_found = [False]
            
        if not self:
            print("General Tree is empty!")
            return grand_parents
    
        if self.data == data:
            is_found[0] = True
            if self.parent:
                if self.parent.parent:
                    grand_parents.append(self.parent.parent.data)
                else:
                    print(f"{data} has no grand parent!")
            else:
                print(f"{data} is a root node and has no parent and grand parent!")
            
        if self.children:
            for child in self.children:
                child.get_all_grand_parent(data, grand_parents, is_found, flag=True)
                
        if not flag and not is_found[0]:
            print(f"No match found for {data}") 
            
        if not grand_parents:
            return
            
        return grand_parents
    
    def get_children_count(self, data):
        children = self.get_children(data)
        if children is not None:
            return len(children)
        return 0
    
    def get_count_key(self, data):
        count = 0
        if not self:
            print("General Tree is empty!")
            return count
        
        if self.data == data:
            return 1
            
        if self.children:
            for child in self.children:
                count += child.get_count_key(data)
                
        return count
    
    def search(self, data, validate: Optional[bool]=False, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if self.data == data:
            if not validate:
                print(f"Match found for {data}")
            return True
            
        if self.children:
            for child in self.children:
                if child.search(data, flag=True):
                    return True
        
        if not flag:     
            if not validate:
                print(f"No match found for {data}")
            return False
    
    def get_first_child(self, children=None):
        if children is None:
            children = []
            
        if not self:
            print("General Tree is empty!")
            return False
        
        if self.children:
            children.append(self.children[0].data)
        
        if self.children:
            for child in self.children:
                child.get_first_child(children)
                
        return children
    
    def get_first_child_of(self, data, flag=False):            
        if not self:
            print("General Tree is empty!")
            return False
        
        if self.data == data:
            if self.children:
                return self.children[0].data
        
        child = None
        if self.children:
            for child in self.children:
                child = child.get_first_child_of(data, flag=True)
                if child:
                    return child
                
        if not flag:
            print(f"No match found for {data}")
            
        return child
    
    def get_middle_child(self, children=None):
        if children is None:
            children = []
            
        if not self:
            print("General Tree is empty!")
            return False
        
        if self.children:
            index = len(self.children)//2
            if len(self.children)%2==0:
                children.append(self.children[index-1].data)
            children.append(self.children[index].data)
                
        
        if self.children:
            for child in self.children:
                child.get_middle_child(children)
                
        return children

    def get_middle_child_of(self, data, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if self.data == data:
            if self.children:
                index = len(self.children)//2
                if len(self.children)%2==0:
                    return [i.data for i in self.children[index-1:index+1]]
                return self.children[index].data
        
        child = None
        if self.children:
            for child in self.children:
                child = child.get_middle_child_of(data, flag=True)
                if child:
                    return child
                
        if not flag:
            print(f"No match found for {data}")
            
        return child
    
    def get_last_child(self, children=None):
        if children is None:
            children = []
            
        if not self:
            print("General Tree is empty!")
            return False
        
        if self.children:
            children.append(self.children[-1].data)
        
        if self.children:
            for child in self.children:
                child.get_last_child(children)
                
        return children
    
    def get_last_child_of(self, data, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if self.data == data:
            if self.children:
                return self.children[-1].data
        
        child = None
        if self.children:
            for child in self.children:
                child = child.get_last_child_of(data, flag=True)
                if child:
                    return child
                
        if not flag:
            print(f"No match found for {data}")
            
        return child
    
    def dfs_traversal(self):
        if not self:
            print("General Tree is empty!")
            return
        
        print(self.data)
        
        if self.children:
            for child in self.children:
                child.dfs_traversal()
                
    def bfs_traversal(self):
        if not self:
            print("General Tree is empty!")
            return
        
        items = [self]
        size = len(items)
        while items:
            for _ in range(size):
                item = items.pop(0)
                print(item.data)
                
            if item.children:
                for child in item.children:
                    items.append(child)
                    
    def insert_sibling_by_value(self, sibling, value, is_inserted=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if is_inserted is None and is_found is None:
            is_inserted = [False]
            is_found = [False] 
             
        if self.get_root(rotation=True) != sibling:  
            if self.children:
                values = [child.data for child in self.children]
                if value not in values:
                    if sibling in values:
                        is_inserted[0] = is_found[0] = True
                        node = GeneralTree(value)
                        self.children.append(node)
                        node.parent = self
                        if self.data == self.get_root(rotation=True):
                            return is_inserted[0]
                        return (is_inserted, is_found)
                else:
                    print(f"Value {value} already exists in the tree!")
                    is_found[0] = True
                    return (is_inserted, is_found)
        else:
            print(f"Match found for {self.get_root(rotation=True)}, but no parent available to add it as sibling")
            return is_inserted[0]
                
        if self.children:
            for child in self.children:
                result = child.insert_sibling_by_value(sibling, value, is_inserted, is_found, flag=True)
                if isinstance(result, tuple):
                    is_inserted, is_found = result
                    if is_inserted[0] or is_found[0]:
                        return is_inserted[0]
                
        if not flag and not is_found[0]:
            print(f"No match found for sibling {sibling}") 
            
        return is_inserted[0]
    
    def insert_sibling_by_value_for_all(self, sibling, value, is_inserted=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if is_inserted is None:
            is_inserted = [False]
            is_found = [False]
        
        if self.get_root(rotation=True) != sibling:
            if self.children:
                values = [child.data for child in self.children]
                if value not in values:
                    if sibling in values:
                        is_inserted[0] = is_found[0] = True
                        node = GeneralTree(value)
                        self.children.append(node)
                        node.parent = self
                else:
                    print(f"Value {value} already exists in the tree!")
                    is_found[0] = True
        else:
            print(f"Match found for {self.get_root(rotation=True)}, but no parent available to add it as sibling")
            is_found[0] = True
                
        if self.children:
            for child in self.children:
                child.insert_sibling_by_value_for_all(sibling, value, is_inserted, is_found, flag=True)
                
        if not flag and not is_found[0]:
            print(f"No match found for sibling {sibling}") 
            
        return is_inserted[0]
    
    def insert_sibling_by_parent_value(self, parent, value, is_inserted=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if is_inserted is None and is_found is None:
            is_inserted = [False]
            is_found = [False]

        if self.data == parent:
            is_found[0] = True
            values = [child.data for child in self.children]
            if value not in values:
                is_inserted[0] = True
                node = GeneralTree(value)
                self.children.append(node)
                node.parent = self
            else:
                print(f"Value {value} already exists in the tree!")

            if self.get_root(rotation=True) == parent:
                return is_inserted[0]
            return (is_inserted, is_found)
            
        if self.children:
            for child in self.children:
                result = child.insert_sibling_by_parent_value(parent, value, is_inserted, is_found, flag=True)
                if isinstance(result, tuple):
                    is_inserted, is_found = result
                if is_inserted[0] or is_found[0]:
                    return is_inserted[0]
            
        if not flag and not is_found[0]:
            print(f"No match found for parent {parent}") 
        
        return is_inserted[0]
    
    def insert_sibling_by_parent_value_for_all(self, parent, value, is_inserted=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if is_inserted is None:
            is_inserted = [False]
            is_found = [False]
        
        if self.data == parent:
            values = [child.data for child in self.children]
            is_found[0] = True
            if value not in values:
                is_inserted[0] = True   
                node = GeneralTree(value)
                self.children.append(node)
                node.parent = self
            else:
                print(f"Value {value} already exists in the tree!")
            
        if self.children:
            for child in self.children:
                child.insert_sibling_by_parent_value_for_all(parent, value, is_inserted, is_found, flag=True)
            
        if not flag and not is_found[0]:
            print(f"No match found for parent {parent}") 
        
        return is_inserted[0]

    def insert_sibling_by_grand_parent_and_parent_value(self, grand_parent, parent, value, is_inserted=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if is_inserted is None and is_found is None:
            is_inserted = [False]
            is_found = [False]
        
        if self.data == grand_parent:
            is_found[0] = True
            if self.children:
                for child in self.children:
                    if child.data == parent:
                        values = [child.data for child in child.children]
                        if value not in values:
                            is_inserted[0] = True
                            node = GeneralTree(value)
                            child.children.append(node)
                            node.parent = child
                        else:
                            print(f"Value {value} already exists in the tree!")
                            
                        if self.get_root(rotation=True) == grand_parent:
                            return is_inserted[0]
                        return (is_inserted, is_found)

            print("Grand parent match found, but not for parent!")
            if self.get_root(rotation=True) == grand_parent:
                return is_inserted[0]
            return (is_inserted, is_found)
            
        if self.children:
            for child in self.children:
                result = child.insert_sibling_by_grand_parent_and_parent_value(grand_parent, parent, value, is_inserted, is_found, flag=True)
                if isinstance(result, tuple):
                    is_inserted, is_found = result
                if is_inserted[0] or is_found[0]:
                    return is_inserted[0]
            
        if not flag and not is_found[0]:
            print(f"No match found for grand parent {grand_parent}") 
            
        return is_inserted[0]
    
    def insert_sibling_by_grand_parent_and_parent_value_for_all(self, grand_parent, parent, value, is_inserted=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if is_inserted is None and is_found is None:
            is_inserted = [False]
            is_found = [False]
        
        if self.data == grand_parent:
            is_found[0] = True
            if self.children:
                for child in self.children:
                    if child.data == parent:
                        values = [child.data for child in child.children]
                        if value not in values:
                            is_inserted[0] = True
                            node = GeneralTree(value)
                            child.children.append(node)
                            node.parent = child
                        else:
                            print(f"Value {value} already exists in the tree!")
                        return

            print("Grand parent match found, but not for parent!")
            
        if self.children:
            for child in self.children:
                child.insert_sibling_by_grand_parent_and_parent_value_for_all(grand_parent, parent, value, is_inserted, is_found, flag=True)
            
        if not flag and not is_found[0]:
            print(f"No match found for grand parent {grand_parent}") 
            
        return is_inserted[0]

    def insert_at_under_parent(self, parent, index, value, is_inserted=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if is_inserted is None and is_found is None:
            is_inserted = [False]
            is_found = [False]
        
        if self.data == parent:
            is_found[0] = True
            if index <= len(self.children):
                values = [child.data for child in self.children]
                if value not in values:
                    is_inserted[0] = True
                    node = GeneralTree(value)
                    self.children.insert(index, node)
                    node.parent = self
                else:
                    print(f"Value {value} already exists in the tree!")
            else:
                print(f"Parent {parent} matched, but invalid Index!")
                
            if self.get_root(rotation=True) == parent:
                return is_inserted[0]
            return (is_inserted, is_found)
        
        if self.children:
            for child in self.children:
                result = child.insert_at_under_parent(parent, index, value, is_inserted, is_found, flag=True)
                if isinstance(result, tuple):
                    is_inserted, is_found = result
                if is_inserted[0] or is_found[0]:
                    return is_inserted[0]
                
        if not flag and not is_found[0]:
            print(f"No match found for parent {parent}")
            
        return is_inserted[0]
    
    def insert_at_under_parent_for_all(self, parent, index, value, is_inserted=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        if is_inserted is None and is_found is None:
            is_inserted = [False]
            is_found = [False]
        
        if self.data == parent:
            is_found[0] = True
            if index <= len(self.children):
                values = [child.data for child in self.children]
                if value not in values:
                    is_inserted[0] = True
                    node = GeneralTree(value)
                    self.children.insert(index, node)
                    node.parent = self
                else:
                    print(f"Value {value} already exists in the tree!")
                return
        
            print(f"Parent {parent} matched, but invalid Index!")
        
        if self.children:
            for child in self.children:
                child.insert_at_under_parent_for_all(parent, index, value, is_inserted, is_found, flag=True)
                
        if not flag and not is_found[0]:
            print(f"No match found for parent {parent}")
            
        return is_inserted[0]
    
    def delete_by_value(self, value, flag=False):
        if not self:
            print("General Tree is empty!")
            return False
        
        data = None
        if self.get_root(rotation=True) != value:
            if self.children:
                for index, child in enumerate(self.children):
                    if child.data == value:
                        return self.children.pop(index).data 
        else:
            print(f"Cannot delete the root node {value}")
            return   
        
        if self.children:
            for child in self.children:
                data = child.delete_by_value(value, flag=True)
                if data is not None:
                    return data
                
        if not flag and data is None:
            print(f"No match found for {value}")
    
    def delete_by_value_for_all(self, value, data=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return
        
        if data is None:
            data = []
            
        if is_found is None:
            is_found = [False]
        
        if self.get_root(rotation=True) != value:
            if self.children:
                for index, child in enumerate(self.children):
                    if child.data == value:
                        is_found[0] = True
                        data.append(self.children.pop(index).data)
        else:
            is_found[0] = True
            print(f"Cannot delete the root node {value}")
        
        if self.children:
            for child in self.children:
                child.delete_by_value_for_all(value, data, is_found, flag=True)
                
        if not flag and not is_found[0]:
            print(f"No match found for {value}")
        
        if not data:
            return
                    
        return data
    
    def delete_by_parent_and_value(self, parent, value, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return
        
        if is_found is None:
            is_found = [False]
        
        data = None
        if self.data == parent:
            is_found[0] = True
            if self.children:
                for index, child in enumerate(self.children):
                    if child.data == value:
                        return self.children.pop(index).data
                    
            print(f"Parent {parent} matched, but no children match available for {value}")   
            return data
        
        if self.children:
            for child in self.children:
                data = child.delete_by_parent_and_value(parent, value, is_found, flag=True)
                if is_found[0]:
                    return data
                
        if not flag and not is_found[0]:
            print(f"No match found for parent {parent}")
                    
        return data
    
    def delete_by_parent_and_value_for_all(self, parent, value, data=None, is_found=None, flag=False):        
        if not self:
            print("General Tree is empty!")
            return
        
        if data is None:
            data = []
            
        if is_found is None:
            is_found = [False]
        
        if self.data == parent:
            is_deleted = False
            is_found[0] = True
            if self.children:
                for index, child in enumerate(self.children):
                    if child.data == value:
                        is_deleted = True
                        data.append(self.children.pop(index).data)
                        
            if not is_deleted:
                print(f"Parent {parent} matched, but no children match available for {value}")   
            
        if self.children:
            for child in self.children:
                child.delete_by_parent_and_value_for_all(parent, value, data, is_found, flag=True)
                
        if not flag and not is_found[0]:
            print(f"No match found for parent {parent}")
            
        if not data:
            return
                    
        return data
    
    def delete_at_under_parent(self, parent, index, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return
        
        if is_found is None:
            is_found = [False]
        
        data = None
        if self.data == parent:
            is_found[0] = True
            if self.children:
                if index < len(self.children):
                    return self.children.pop(index).data
               
            print(f"Parent {parent} matched, but invalid Index!")  
            return 
        
        if self.children:
            for child in self.children:
                data = child.delete_at_under_parent(parent, index, is_found, flag=True)
                if data or is_found[0]:
                    return data
                
        if not flag and not is_found[0]:
            print(f"No match found for parent {parent}")
                    
        return data
    
    def delete_at_under_all_parent(self, parent, index, data=None, is_found=None, flag=False):
        if not self:
            print("General Tree is empty!")
            return []
        
        if data is None and is_found is None:
            data = []
            is_found = [False]
        
        if self.data == parent:
            is_found[0] = True
            value = None
            if self.children:
                if index < len(self.children):
                    value = self.children.pop(index).data
                    data.append(value)
                
            if not value:    
                print(f"Parent {parent} matched, but invalid Index!")  
        
        if self.children:
            for child in self.children:
                child.delete_at_under_all_parent(parent, index, data, is_found, flag=True)
                
                
        if not flag and not is_found[0]:
            print(f"No match found for parent {parent}")
            
        if not data:
            return
                    
        return data
        
def fetch_url():
    base_path = os.getcwd()
    list_folder = os.listdir()
    directory = None
    for folder in list_folder:
        if os.path.isdir(folder) and re.match(r'files', folder):
            directory = folder

    file_path = None     
    if directory:
        folder_path = os.path.join(base_path, directory)
        for file in os.listdir(folder_path):
            if os.path.isfile(os.path.join(folder_path, file)) and re.search(r'.*general_tree_[\w\W]+.json', file):
                file_path = os.path.join(folder_path, file)
                return file_path
    else:
        print("No folder available called \"files\"")
        
def read_json():
    file_path = fetch_url()
    if file_path:
        try:
            with open(file_path, 'rt') as f:
                data = json.load(f)
                return data
        except JSONDecodeError:
            print(f"Error: The file {file_path} is not a valid JSON file.")
    else:
        print("No file available called \"general_tree_*.json\"")

def build_tree(root=None, children=None):
    if root is None:
        data = read_json()
        val = list(data.keys())[0]
        root = GeneralTree(val)
        build_tree(root, data[val])
    else:
        for child in children.keys():
            node = GeneralTree(child)
            if isinstance(children[child], dict):
                build_tree(node, children[child])
            elif isinstance(children[child], list):
                for child in children[child]:
                    node.add_child(GeneralTree(child))     
            root.add_child(node)           

    return root

In [632]:

if __name__ == "__main__":
    tree = build_tree()
    tree.print_tree()

Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
      |--Lenovo
        |--i3
        |--i5


In [633]:
print("Root of the General Tree:", tree.get_root())
print("Number of items in General Tree:", tree.count_items())
print("Leaf nodes of the General Tree:", tree.get_leaf_nodes())

Root of the General Tree: Electronics
Number of items in General Tree: 24
Leaf nodes of the General Tree: ['Samsung', 'MI', 'LG', 'Apple', 'Redmi', 's22', 's22 Ultra', 'Air', 'Pro', 'i3', 'i5', 'i3', 'i5']


In [634]:
print("Depth of the General Tree:", tree.get_depth())
print("Height of the General Tree:", tree.get_height())
print("Diameter of the General Tree:", tree.get_diameter())

Depth of the General Tree: 5
Height of the General Tree: 5
Diameter of the General Tree: 9


In [635]:
print("Diameter of the General Tree for Brands:", tree.get_diameter_of("Brands"))
print("Diameter of the General Tree for Specifications:", tree.get_diameter_of("Specifications"))

Diameter of the General Tree for Brands: 5
No match found for Specifications
Diameter of the General Tree for Specifications: None


In [636]:
print(f"Depth of Cell Phone:", tree.depth_of("Cell Phone"))
print(f"Depth of Brands:", tree.depth_of("Brands"))
print(f"Depth of Electronics:", tree.depth_of("Electronics"))
print(f"Depth of Electronics:", tree.depth_of("i3"))
print(f"Depth of Micromax:", tree.depth_of("Micromax"))

Depth of Cell Phone: 1
Depth of Brands: 2
Depth of Electronics: 0
Depth of Electronics: 4
No match found for Micromax
Depth of Micromax: None


In [637]:
print(f"Height of Laptop:", tree.height_of("Laptop"))
print(f"Height of i3:", tree.height_of("i3"))
print(f"Height of TV:", tree.height_of("TV"))
print(f"Height of Electronics:", tree.height_of("Electronics"))
print(f"Height of Micromax:", tree.height_of("Micromax"))

Height of Laptop: 3
Height of i3: 0
Height of TV: 1
Height of Electronics: 4
No match found for Micromax
Height of Micromax: None


In [638]:
print(f"Parent of Electronics:", tree.get_parent("Electronics"))
print(f"Parent of Samsung:", tree.get_parent("Samsung"))
print(f"Parent of Cell Phone:", tree.get_parent("Cell Phone"))
print(f"Parent of Micromax:", tree.get_parent("Micromax"))

Electronics is the root node and has no parent!
Parent of Electronics: None
Parent of Samsung: TV
Parent of Cell Phone: Electronics
No match found for Micromax
Parent of Micromax: None


In [639]:
print(f"Parents of Electronics:", tree.get_parents("Electronics"))
print(f"Parents of Samsung:", tree.get_parents("Samsung"))
print(f"Parents of Cell Phone:", tree.get_parents("Cell Phone"))
print(f"Parents of Brands:", tree.get_parents("Brands"))
print(f"Parents of Micromax:", tree.get_parents("Micromax"))

Electronics is the root node and has no parent!
Parents of Electronics: None
Parents of Samsung: ['TV', 'Brands', 'Brands']
Parents of Cell Phone: ['Electronics']
Parents of Brands: ['Cell Phone', 'Laptop']
No match found for Micromax
Parents of Micromax: None


In [640]:
print(f"Children of Electronics:", tree.get_children("Electronics"))
print(f"Children of Samsung:", tree.get_children("Samsung"))
print(f"Children of Cell Phone:", tree.get_children("Cell Phone"))
print(f"Children of Micromax:", tree.get_children("Micromax"))

Children of Electronics: ['TV', 'Cell Phone', 'Laptop']
Match found, but no child available for Samsung!
Children of Samsung: None
Children of Cell Phone: ['Brands']
No match found for Micromax
Children of Micromax: None


In [641]:
print(f"Children of Electronics:", tree.get_all_children("Electronics"))
print(f"Children of Samsung:", tree.get_all_children("Samsung"))
print(f"Children of Cell Phone:", tree.get_all_children("Cell Phone"))
print(f"Children of Micromax:", tree.get_all_children("Micromax"))

Children of Electronics: ['TV', 'Cell Phone', 'Laptop']
Match found, but no child available for Samsung!
Children of Samsung: ['s22', 's22 Ultra', 'i3', 'i5']
Children of Cell Phone: ['Brands']
No match found for Micromax
Children of Micromax: None


In [642]:
print(f"Grand children of Electronics:", tree.get_grand_children("Electronics"))
print(f"Grand children of Samsung:", tree.get_grand_children("Samsung"))
print(f"Grand children of Brands:", tree.get_grand_children("Brands"))
print(f"Grand children of Micromax:", tree.get_grand_children("Micromax"))

Grand children of Electronics: ['Samsung', 'MI', 'LG']
Match found, but no children and grand children available for Samsung!
Grand children of Samsung: None
Match found, but no grand children available for Brands!
Grand children of Brands: None
No match found for Micromax
Grand children of Micromax: None


In [643]:
print(f"Grand children of Electronics:", tree.get_all_grand_children("Electronics"))
print(f"Grand children of Samsung:", tree.get_all_grand_children("Samsung"))
print(f"Grand children of Brands:", tree.get_all_grand_children("Brands"))
print(f"Grand children of Micromax:", tree.get_all_grand_children("Micromax"))

Grand children of Electronics: ['Samsung', 'MI', 'LG', 'Brands', 'Brands']
Match found, but no children and grand children available for Samsung!
Match found, but no grand children available for Samsung!
Match found, but no grand children available for Samsung!
Grand children of Samsung: None
Grand children of Brands: ['Redmi', 's22', 's22 Ultra', 'Air', 'Pro', 'i3', 'i5', 'i3', 'i5']
No match found for Micromax
Grand children of Micromax: None


In [644]:
print(f"Grand parent of Electronics:", tree.get_grand_parent("Electronics"))
print(f"Grand parent of Samsung:", tree.get_grand_parent("Samsung"))
print(f"Grand parent of Cell Phone:", tree.get_grand_parent("Cell Phone"))
print(f"Grand parent of s22:", tree.get_grand_parent("s22"))
print(f"Grand parent of Micromax:", tree.get_grand_parent("Micromax"))

Electronics is a root node and has no parent and grand parent!
Grand parent of Electronics: None
Grand parent of Samsung: Electronics
Cell Phone has no grand parent!
Grand parent of Cell Phone: None
Grand parent of s22: Brands
No match found for Micromax
Grand parent of Micromax: None


In [645]:
print(f"Grand parents of Electronics:", tree.get_all_grand_parent("Electronics"))
print(f"Grand parents of Samsung:", tree.get_all_grand_parent("Samsung"))
print(f"Grand parents of Cell Phone:", tree.get_all_grand_parent("Cell Phone"))
print(f"Grand parents of s22:", tree.get_all_grand_parent("s22"))
print(f"Grand parents of Micromax:", tree.get_all_grand_parent("Micromax"))

Electronics is a root node and has no parent and grand parent!
Grand parents of Electronics: None
Grand parents of Samsung: ['Electronics', 'Cell Phone', 'Laptop']
Cell Phone has no grand parent!
Grand parents of Cell Phone: None
Grand parents of s22: ['Brands']
No match found for Micromax
Grand parents of Micromax: None


In [646]:
print(f"Count of children for Electronics:", tree.get_children_count("Electronics"))
print(f"Count of children for Samsung:", tree.get_children_count("Samsung"))
print(f"Count of children for TV:", tree.get_children_count("TV"))
print(f"Count of children for Laptop:", tree.get_children_count("Laptop"))
print(f"Count of children for Micromax:", tree.get_children_count("Micromax"))

Count of children for Electronics: 3
Match found, but no child available for Samsung!
Count of children for Samsung: 0
Count of children for TV: 3
Count of children for Laptop: 1
No match found for Micromax
Count of children for Micromax: 0


In [647]:
print(f"Count of node containing key as TV:", tree.get_count_key("TV"))
print(f"Count of node containing key as Samsung:", tree.get_count_key("Samsung"))    
print(f"Count of node containing key as Micromax:", tree.get_count_key("Micromax"))    

Count of node containing key as TV: 1
Count of node containing key as Samsung: 3
Count of node containing key as Micromax: 0


In [648]:
print(f"Search for key Samsung:", tree.search("Samsung"))
print(f"Search for key TV:", tree.search("TV"))
print(f"Search for key Google Pixel:", tree.search("Google Pixel"))

Match found for Samsung
Search for key Samsung: True
Match found for TV
Search for key TV: True
No match found for Google Pixel
Search for key Google Pixel: False


In [649]:

print("The first child of each parent:", tree.get_first_child())

The first child of each parent: ['TV', 'Samsung', 'Brands', 'Apple', 'Redmi', 's22', 'Brands', 'Apple', 'Air', 'i3', 'i3']


In [650]:
print("The first child of parent 'Electronics':", tree.get_first_child_of("Electronics"))
print("The first child of parent 'Cell Phone':", tree.get_first_child_of("Cell Phone"))
print("The first child of parent 'Samsung':", tree.get_first_child_of("Samsung"))
print("The first child of parent 'Micromax':", tree.get_first_child_of("Micromax"))

The first child of parent 'Electronics': TV
The first child of parent 'Cell Phone': Brands
The first child of parent 'Samsung': s22
No match found for Micromax
The first child of parent 'Micromax': None


In [651]:
print("The middle child of each parent:", tree.get_middle_child())

The middle child of each parent: ['Cell Phone', 'MI', 'Brands', 'MI', 'Redmi', 's22', 's22 Ultra', 'Brands', 'Samsung', 'Air', 'Pro', 'i3', 'i5', 'i3', 'i5']


In [652]:
print("The middle child of parent 'Electronics':", tree.get_middle_child_of("Electronics"))
print("The middle child of parent 'Cell Phone':", tree.get_middle_child_of("Cell Phone"))
print("The middle child of parent 'Samsung':", tree.get_middle_child_of("Samsung"))
print("The middle child of parent 'Micromax':", tree.get_middle_child_of("Micromax"))

The middle child of parent 'Electronics': Cell Phone
The middle child of parent 'Cell Phone': Brands
The middle child of parent 'Samsung': ['s22', 's22 Ultra']
No match found for Micromax
The middle child of parent 'Micromax': None


In [653]:
print("The last child of each parent:", tree.get_last_child())

The last child of each parent: ['Laptop', 'LG', 'Brands', 'Samsung', 'Redmi', 's22 Ultra', 'Brands', 'Lenovo', 'Pro', 'i5', 'i5']


In [654]:
print("The last child of parent 'Electronics':", tree.get_last_child_of("Electronics"))
print("The last child of parent 'Cell Phone':", tree.get_last_child_of("Cell Phone"))
print("The last child of parent 'Samsung':", tree.get_last_child_of("Samsung"))
print("The last child of parent 'Micromax':", tree.get_last_child_of("Micromax"))

The last child of parent 'Electronics': Laptop
The last child of parent 'Cell Phone': Brands
The last child of parent 'Samsung': s22 Ultra
No match found for Micromax
The last child of parent 'Micromax': None


In [655]:
print(f"Depth First Search:")
tree.dfs_traversal()

Depth First Search:
Electronics
TV
Samsung
MI
LG
Cell Phone
Brands
Apple
MI
Redmi
Samsung
s22
s22 Ultra
Laptop
Brands
Apple
Air
Pro
Samsung
i3
i5
Lenovo
i3
i5


In [656]:
print(f"Breadth First Search:")
tree.bfs_traversal()

Breadth First Search:
Electronics
TV
Cell Phone
Laptop
Samsung
MI
LG
Brands
Brands
Apple
MI
Samsung
Apple
Samsung
Lenovo
Redmi
s22
s22 Ultra
Air
Pro
i3
i5
i3
i5


In [657]:
print(tree.insert_sibling_by_value("Cell Phone", "Desktop"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
      |--Lenovo
        |--i3
        |--i5
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 25
Diameter of the General Tree: 9


In [658]:
print(tree.insert_sibling_by_value("Electronics", "Electrical"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Match found for Electronics, but no parent available to add it as sibling
False
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
      |--Lenovo
        |--i3
        |--i5
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 25
Diameter of the General Tree: 9


In [659]:
print(tree.insert_sibling_by_value("Android", "iOS"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for sibling Android
False
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
      |--Lenovo
        |--i3
        |--i5
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 25
Diameter of the General Tree: 9


In [660]:
print(tree.insert_sibling_by_value("Brands", "Specifications"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
      |--Lenovo
        |--i3
        |--i5
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 26
Diameter of the General Tree: 9


In [661]:
print(tree.insert_sibling_by_value("Brands", "Specifications"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value Specifications already exists in the tree!
False
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
      |--Lenovo
        |--i3
        |--i5
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 26
Diameter of the General Tree: 9


In [662]:
print(tree.insert_sibling_by_value("Samsung", "Hitachi"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
      |--Lenovo
        |--i3
        |--i5
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 27
Diameter of the General Tree: 9


In [663]:
print(tree.insert_sibling_by_value_for_all("Android", "iOS"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for sibling Android
False
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
      |--Lenovo
        |--i3
        |--i5
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 27
Diameter of the General Tree: 9


In [664]:
print(tree.insert_sibling_by_value_for_all("i3", "i7"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 29
Diameter of the General Tree: 9


In [665]:
print(tree.insert_sibling_by_value_for_all("Electronics", "Electrical"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Match found for Electronics, but no parent available to add it as sibling
False
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 29
Diameter of the General Tree: 9


In [666]:
print(tree.insert_sibling_by_value_for_all("Brands", "Specifications"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value Specifications already exists in the tree!
True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
    |--Specifications
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 30
Diameter of the General Tree: 9


In [667]:
print(tree.insert_sibling_by_parent_value("Cell Phone", "Features"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
    |--Specifications
  |--Desktop
Depth of the General Tree: 5
Number of items in General Tree: 31
Diameter of the General Tree: 9


In [668]:
print(tree.insert_sibling_by_parent_value("Electronics", "Home Appliances"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
    |--Specifications
  |--Desktop
  |--Home Appliances
Depth of the General Tree: 5
Number of items in General Tree: 32
Diameter of the General Tree: 9


In [669]:
print(tree.insert_sibling_by_parent_value("Electronics", "Home Appliances"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value Home Appliances already exists in the tree!
False
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
    |--Specifications
  |--Desktop
  |--Home Appliances
Depth of the General Tree: 5
Number of items in General Tree: 32
Diameter of the General Tree: 9


In [670]:
print(tree.insert_sibling_by_parent_value("Android", "JellyBean"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for parent Android
False
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
    |--Specifications
  |--Desktop
  |--Home Appliances
Depth of the General Tree: 5
Number of items in General Tree: 32
Diameter of the General Tree: 9


In [671]:
print(tree.insert_sibling_by_parent_value_for_all("Electronics", "Office Supplies"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 33
Diameter of the General Tree: 9


In [672]:
print(tree.insert_sibling_by_parent_value_for_all("Brands", "Sony"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 35
Diameter of the General Tree: 9


In [673]:
print(tree.insert_sibling_by_parent_value_for_all("Electronics", "Office Supplies"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value Office Supplies already exists in the tree!
False
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 35
Diameter of the General Tree: 9


In [674]:
print(tree.insert_sibling_by_parent_value_for_all("MI", "Redmi"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value Redmi already exists in the tree!
True
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 36
Diameter of the General Tree: 9


In [675]:
print(tree.insert_sibling_by_grand_parent_and_parent_value("Cell Phone", "Apple", "Pro"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Grand parent match found, but not for parent!
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 36
Diameter of the General Tree: 9


In [676]:
print(tree.insert_sibling_by_grand_parent_and_parent_value("Brands", "Apple", "Pro"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 37
Diameter of the General Tree: 9


In [677]:
print(tree.insert_sibling_by_grand_parent_and_parent_value("i8", "GPU", "16gb"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for grand parent i8
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 37
Diameter of the General Tree: 9


In [678]:
print(tree.insert_sibling_by_grand_parent_and_parent_value("Brands", "Apple", "Pro"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value Pro already exists in the tree!
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 37
Diameter of the General Tree: 9


In [679]:
print(tree.insert_sibling_by_grand_parent_and_parent_value_for_all("Apple", "Pro", "GPU"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 39
Diameter of the General Tree: 11


In [680]:
print(tree.insert_sibling_by_grand_parent_and_parent_value_for_all("Apple", "Pro", "GPU"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value GPU already exists in the tree!
Value GPU already exists in the tree!
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 39
Diameter of the General Tree: 11


In [681]:
print(tree.insert_sibling_by_grand_parent_and_parent_value_for_all("i8", "GPU", "16gb"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for grand parent i8
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 39
Diameter of the General Tree: 11


In [682]:
print(tree.insert_sibling_by_grand_parent_and_parent_value_for_all("Cell Phone", "Apple", "Pro"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Grand parent match found, but not for parent!
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 39
Diameter of the General Tree: 11


In [683]:
print(tree.insert_at_under_parent("Electronics", 1, "Keyboard"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Keyboard
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 40
Diameter of the General Tree: 11


In [684]:
print(tree.insert_at_under_parent("Cell Phone", 5, "Blueberry"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Parent Cell Phone matched, but invalid Index!
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Keyboard
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 40
Diameter of the General Tree: 11


In [685]:
print(tree.insert_at_under_parent("Cell Phone", 0, "Brands"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value Brands already exists in the tree!
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Keyboard
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 40
Diameter of the General Tree: 11


In [686]:
print(tree.insert_at_under_parent_for_all("Cell Phone", 0, "Brands"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Value Brands already exists in the tree!
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Keyboard
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 40
Diameter of the General Tree: 11


In [687]:
print(tree.insert_at_under_parent_for_all("Cell Phone", 8, "Blueberry"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Parent Cell Phone matched, but invalid Index!
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Keyboard
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 40
Diameter of the General Tree: 11


In [688]:
print(tree.insert_at_under_parent_for_all("Brands", 1, "Motorola"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

True
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Keyboard
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 42
Diameter of the General Tree: 11


In [689]:
print(tree.insert_at_under_parent_for_all("Hardwares", 1, "Motorola"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for parent Hardwares
False
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Keyboard
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 42
Diameter of the General Tree: 11


In [690]:
print(tree.delete_by_value("Electronics"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Cannot delete the root node Electronics
None
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Keyboard
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 42
Diameter of the General Tree: 11


In [691]:
print(tree.delete_by_value("Keyboard"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Keyboard
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 41
Diameter of the General Tree: 11


In [692]:
print(tree.delete_by_value("Electrical"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for Electrical
None
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 41
Diameter of the General Tree: 11


In [693]:
print(tree.delete_by_value_for_all("Electrical"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for Electrical
None
Electronics
  |--TV
    |--Samsung
    |--MI
      |--Redmi
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
        |--Redmi
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 41
Diameter of the General Tree: 11


In [694]:
print(tree.delete_by_value_for_all("Redmi"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

['Redmi', 'Redmi']
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 39
Diameter of the General Tree: 11


In [695]:
print(tree.delete_by_value_for_all("Electronics"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Cannot delete the root node Electronics
None
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
      |--Samsung
        |--s22
        |--s22 Ultra
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 39
Diameter of the General Tree: 11


In [696]:
print(tree.delete_by_parent_and_value("Brands", "Samsung"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())   

Samsung
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 36
Diameter of the General Tree: 11


In [697]:
print(tree.delete_by_parent_and_value("LG", "32 in"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Parent LG matched, but no children match available for 32 in
None
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 36
Diameter of the General Tree: 11


In [698]:
print(tree.delete_by_parent_and_value("Micromax", "32 in"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for parent Micromax
None
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
        |--Pro
          |--GPU
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
        |--Pro
          |--GPU
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 6
Number of items in General Tree: 36
Diameter of the General Tree: 11


In [699]:
print(tree.delete_by_parent_and_value_for_all("Apple", "Pro"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())   

['Pro', 'Pro']
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 32
Diameter of the General Tree: 8


In [700]:
print(tree.delete_by_parent_and_value_for_all("LG", "32 in"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

Parent LG matched, but no children match available for 32 in
None
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 32
Diameter of the General Tree: 8


In [701]:
print(tree.delete_by_parent_and_value_for_all("Micromax", "32 in"))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

No match found for parent Micromax
None
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 32
Diameter of the General Tree: 8


In [702]:
print(tree.delete_at_under_parent("Micromax", 2))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter()) 

No match found for parent Micromax
None
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 32
Diameter of the General Tree: 8


In [703]:
print(tree.delete_at_under_parent("Apple", 2))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter()) 

Parent Apple matched, but invalid Index!
None
Electronics
  |--TV
    |--Samsung
    |--MI
    |--LG
    |--Hitachi
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 32
Diameter of the General Tree: 8


In [704]:
print(tree.delete_at_under_parent("Electronics", 0))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter()) 

TV
Electronics
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 27
Diameter of the General Tree: 8


In [705]:
print(tree.delete_at_under_all_parent("Sony", 3))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())  

Parent Sony matched, but invalid Index!
Parent Sony matched, but invalid Index!
None
Electronics
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 27
Diameter of the General Tree: 8


In [706]:
print(tree.delete_at_under_all_parent("Micromax", 2))
tree.print_tree()       
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter()) 

No match found for parent Micromax
None
Electronics
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--MI
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Samsung
        |--i3
        |--i5
        |--i7
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 27
Diameter of the General Tree: 8


In [707]:
print(tree.delete_at_under_all_parent("Brands", 2))
tree.print_tree()
print("Depth of the General Tree:", tree.get_depth())
print("Number of items in General Tree:", tree.count_items())
print("Diameter of the General Tree:", tree.get_diameter())

['MI', 'Samsung']
Electronics
  |--Cell Phone
    |--Brands
      |--Apple
      |--Motorola
      |--Sony
    |--Specifications
    |--Features
  |--Laptop
    |--Brands
      |--Apple
        |--Air
      |--Motorola
      |--Lenovo
        |--i3
        |--i5
        |--i7
      |--Sony
    |--Specifications
  |--Desktop
  |--Home Appliances
  |--Office Supplies
Depth of the General Tree: 5
Number of items in General Tree: 22
Diameter of the General Tree: 8
